# EpiGraph on Kaggle
Bật **GPU** + **Internet** và add dataset chứa 2 file `.pkl`. Chạy lần lượt từ trên xuống. Cell nào cũng chạy lại được (idempotent).

In [ ]:
# 1. Clone repo (chỉ 1 lần), chép data, cài torch_geometric
import os, glob, shutil, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("EpiGraph"):
    subprocess.run(["git", "clone", "https://github.com/sj584/EpiGraph"], check=True)
os.chdir("/kaggle/working/EpiGraph")

pkls = glob.glob("/kaggle/input/**/*.pkl", recursive=True)
print("pkl tim thay:", pkls)
assert pkls, "Khong thay file .pkl - hay Add Input dataset chua 2 file pkl"
for p in pkls:
    shutil.copy(p, ".")
print(os.listdir("."))
print("checkpoint:", sorted(os.listdir("checkpoint")))


In [ ]:
!pip install -q torch_geometric

In [ ]:
# 2. Va code cho torch_geometric ban moi (chay lai nhieu lan cung an toan)
import re, os
os.chdir("/kaggle/working/EpiGraph")

# model.py: data.node_attrs bi trung ten voi method cua PyG moi
s = open("model.py").read()
s = s.replace("data.node_attrs", "data['node_attrs']")
open("model.py", "w").write(s)

# evaluate.py: doi ten key lin_src -> lin khi load checkpoint cu
s = open("evaluate.py").read()
if "lin_src" not in s:
    pat = re.compile(r"^([ \t]*)model\.load_state_dict\(torch\.load\(f'\{direc\}/\{pt\}'\)\)", re.M)
    assert pat.search(s), "khong tim thay dong load_state_dict trong evaluate.py"
    def rep(m):
        i = m.group(1)
        return (i + "sd = torch.load(f'{direc}/{pt}', map_location='cpu')\n"
                + i + "sd = {k.replace('lin_src', 'lin'): v for k, v in sd.items() if 'lin_dst' not in k}\n"
                + i + "model.load_state_dict(sd)")
    s = pat.sub(rep, s)
    open("evaluate.py", "w").write(s)

# train.py: Data cua PyG moi khong hash duoc -> khong dung set()
s = open("train.py").read()
old = "X_train = list(set(X) - set(X_test))"
if old in s:
    s = s.replace(old, "X_train = [d for i, d in enumerate(X) if i < test_sta or i > test_end]")
    open("train.py", "w").write(s)

print("OK")
!grep -n "node_attrs" model.py
!grep -n "lin_src" evaluate.py
!grep -n "X_train =" train.py


## 3. Evaluate bằng checkpoint của tác giả (nhanh, không cần train)
Kỳ vọng theo bài: AUC-ROC ~0.73, F1 ~0.29, MCC ~0.19, BACC ~0.62.

In [ ]:
import os
os.chdir("/kaggle/working/EpiGraph")
assert any(f.endswith(".pt") for f in os.listdir("checkpoint")), "checkpoint/ khong co file .pt"
!python evaluate.py

## 4. Train lại (10 fold x 50 epoch, tốn thời gian)
Chạy thử 1 lần xem epoch đầu mất bao lâu rồi mới quyết định.

In [ ]:
import os
os.chdir("/kaggle/working/EpiGraph")
os.makedirs("models", exist_ok=True)
!python -u train.py --model_save_dir models

## 5. Evaluate model vừa train

In [ ]:
import os
os.chdir("/kaggle/working/EpiGraph")
assert any(f.endswith(".pt") for f in os.listdir("models")), "models/ chua co file .pt - train chua xong?"
!python evaluate.py --model_checkpoint models